<a href="https://colab.research.google.com/github/yanxinlii/decision-tree-classifier/blob/main/customer_segmentation_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##1. IMPORTS & SETUP

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# For the black-box comparison and evaluation utilities
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder

# Reproducibility seed — fix this so results are consistent across runs
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("All libraries imported successfully.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. DATA LOADING & EXPLORATORY DATA ANALYSIS (EDA)

In [ ]:
# ── 2.1  Load ────────────────────────────────────────────────────────────────
df = pd.read_csv("Forest_Cover_Type_Prediction_train.csv")

# Drop the Id column — it is a row identifier, not a predictive feature
df.drop(columns=["Id"], inplace=True)

print(f"Dataset shape (rows × columns): {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

## Dataset Overview

The Forest Cover Type dataset contains **15,120 samples and 55 features** (after dropping the `Id` column).
Features fall into three groups: 10 continuous topographic measurements (e.g., Elevation, Slope,
Hillshade readings), 4 binary Wilderness Area indicators, and 40 binary Soil Type indicators.
The target variable `Cover_Type` has 7 classes representing different forest cover types.
Notably, there are **zero missing values**, so no imputation is needed.

In [ ]:
# ── 2.2  Basic Statistics ─────────────────────────────────────────────────────
print("=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
print(df.isnull().sum().sum(), "total missing values")

print("\n=== Descriptive Statistics (continuous features) ===")
continuous_cols = ["Elevation","Aspect","Slope",
                   "Horizontal_Distance_To_Hydrology","Vertical_Distance_To_Hydrology",
                   "Horizontal_Distance_To_Roadways",
                   "Hillshade_9am","Hillshade_Noon","Hillshade_3pm",
                   "Horizontal_Distance_To_Fire_Points"]
df[continuous_cols].describe().T

## Descriptive Statistics Insights

From the continuous feature statistics:
- **Elevation** ranges from 1,863m to 3,849m with a mean of ~2,749m, suggesting the dataset
  covers a wide vertical range of the Roosevelt National Forest in Colorado.
- **Aspect** (compass bearing of slope face) spans the full 0°–360° range with high variance
  (std ≈ 110), indicating diverse terrain orientations across samples.
- **Vertical_Distance_To_Hydrology** has negative values (min = –146), meaning some points
  are *below* nearby water sources — a physically meaningful edge case the model must handle.
- **Hillshade** values (9am, Noon, 3pm) show moderate standard deviations, capturing the
  variation in solar exposure throughout the day across different terrain aspects and slopes.
- All features are already on integer scales with no normalization required for tree-based
  methods, since decision trees split on thresholds and are invariant to monotonic scaling.

In [ ]:
# ── 2.3  Target Distribution ──────────────────────────────────────────────────
# Cover_Type labels: 1=Spruce/Fir, 2=Lodgepole Pine, 3=Ponderosa Pine,
#                    4=Cottonwood/Willow, 5=Aspen, 6=Douglas-fir, 7=Krummholz
cover_type_names = {
    1: "Spruce/Fir",
    2: "Lodgepole Pine",
    3: "Ponderosa Pine",
    4: "Cottonwood/Willow",
    5: "Aspen",
    6: "Douglas-fir",
    7: "Krummholz"
}

counts = df["Cover_Type"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([cover_type_names[i] for i in counts.index], counts.values,
       color=sns.color_palette("Set2", 7), edgecolor="black")
ax.set_title("Forest Cover Type Distribution (Perfectly Balanced)", fontsize=14, fontweight="bold")
ax.set_xlabel("Cover Type")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("target_distribution.png", dpi=150)
plt.show()
print("Each class has exactly 2,160 samples → perfectly balanced dataset.")

## Class Balance

Each of the 7 cover types contains exactly **2,160 samples**, making this a **perfectly balanced
dataset**. This is highly favorable for classification: accuracy is a fair and unbiased metric
(no class will dominate predictions by sheer frequency), and no class-weighting or oversampling
techniques such as SMOTE are necessary. This also means macro-averaged and weighted-averaged
metrics will be equivalent in the evaluation results.

In [ ]:
# ── 2.4  Feature Correlation Heatmap (continuous features only) ───────────────
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[continuous_cols + ["Cover_Type"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Correlation Heatmap — Continuous Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.show()

## Correlation Insights

Several notable patterns emerge from the heatmap:
- **Hillshade_9am and Hillshade_3pm** are strongly negatively correlated (–0.78), which is
  physically expected — a slope facing east receives morning sun but is shaded in the afternoon,
  and vice versa for west-facing slopes.
- **Hillshade_Noon and Slope** show a moderate negative correlation (–0.61), as steeper slopes
  receive less direct overhead sunlight at noon.
- **Elevation and Horizontal_Distance_To_Roadways** are positively correlated (0.58), suggesting
  roads are generally built at lower elevations and higher-elevation areas are more remote.
- **Horizontal and Vertical Distance to Hydrology** are correlated (0.65), which is geometrically
  expected — points further horizontally from water tend to also be further vertically.
- **Cover_Type shows very low linear correlations with all individual features** (all near 0),
  which suggests that the classification boundary is highly non-linear — motivating the use of
  a decision tree that can capture complex, non-linear feature interactions through recursive
  partitioning.

In [ ]:
# ── 2.5  Distribution of key continuous features by Cover Type ────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
key_features = ["Elevation", "Slope", "Aspect",
                "Horizontal_Distance_To_Hydrology",
                "Horizontal_Distance_To_Roadways",
                "Horizontal_Distance_To_Fire_Points"]

palette = sns.color_palette("Set2", 7)
for ax, feat in zip(axes.flatten(), key_features):
    for ct in sorted(df["Cover_Type"].unique()):
        subset = df[df["Cover_Type"] == ct][feat]
        ax.hist(subset, bins=30, alpha=0.5, label=cover_type_names[ct], color=palette[ct - 1])
    ax.set_title(feat, fontsize=10, fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")

# Single shared legend below the plots
handles = [mpatches.Patch(color=palette[i], label=cover_type_names[i + 1]) for i in range(7)]
fig.legend(handles=handles, loc="lower center", ncol=4, fontsize=9, title="Cover Type")
plt.suptitle("Feature Distributions by Cover Type", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("feature_distributions.png", dpi=150)
plt.show()

## Feature Distribution Insights by Cover Type

- **Elevation** is the most visually discriminative feature: Krummholz (Type 7) clusters at
  very high elevations (around 3,500m+), while Cottonwood/Willow (Type 4) occupies distinctly lower
  elevations (around 2,200m). This explains why Elevation becomes the root split in the tree
  (Section 10).
- **Slope** distributions largely overlap across classes, suggesting it is a weaker individual
  discriminator — though it may still contribute in combination with other features.
- **Horizontal Distance to Roadways** shows that Cottonwood/Willow tends to appear closer to
  roads, likely because this species favors riparian zones near accessible valley floors.
- **Horizontal Distance to Fire Points** varies substantially across types, which may reflect
  historical fire management patterns tied to specific vegetation zones.
- The heavy overlap in most features across classes confirms that no single feature is
  sufficient for classification, reinforcing why the tree needs sufficient depth (tuned to 14
  in Section 7) to form accurate composite decision rules.

##3. DATA PREPROCESSING & TRAIN/TEST SPLIT

In [ ]:
# ── 3.1  Separate features and target ─────────────────────────────────────────
X = df.drop(columns=["Cover_Type"]).values   # numpy array: (15120, 54)
y = df["Cover_Type"].values                  # numpy array: (15120,)

# Store feature names for later interpretability analysis
feature_names = df.drop(columns=["Cover_Type"]).columns.tolist()
class_names   = [cover_type_names[i] for i in sorted(np.unique(y))]

print(f"Feature matrix X: {X.shape}")
print(f"Target vector  y: {y.shape}")
print(f"Classes: {np.unique(y)}")

In [ ]:
# ── 3.2  Train / Validation / Test split (70 / 15 / 15) ──────────────────────
# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)
# Second split: temp → 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"Train size : {X_train.shape[0]:>5}  ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val   size : {X_val.shape[0]:>5}  ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test  size : {X_test.shape[0]:>5}  ({X_test.shape[0]/len(X)*100:.1f}%)")

## Optimization Problem: CART as Greedy Binary Minimization

### Problem Formulation

At the heart of a CART decision tree is a **recursive optimization problem**. At each internal
node $t$, given the set of training samples $S_t = \{(x_i, y_i)\}$ that reach that
node, we search for the binary split that minimizes the weighted Gini impurity of the two
resulting child nodes:

$$\min_{j \in \{1,\dots,p\},\ \theta \in T_j} \quad \frac{n_L}{n_t} \cdot \text{Gini}(S_L) + \frac{n_R}{n_t} \cdot \text{Gini}(S_R)$$

where:
- $j$ is the chosen feature index, $\theta$ is the split threshold
- $S_L = \{(x_i, y_i) \in S_t : x_{ij} \leq \theta\}$ and
  $S_R = S_t \setminus S_L$ are the left and right child sets
- $n_t = |S_t|$, $n_L = |S_L|$, $n_R = |S_R|$
- The **Gini impurity** of a node with class proportions $p_1, \dots, p_K$ is defined as:

$$\text{Gini}(S) = 1 - \sum_{k=1}^{K} p_k^2$$

A perfectly pure node (all samples belong to one class) has Gini = 0. A maximally impure node
(uniform distribution across all $K$ classes) approaches $1 - \frac{1}{K}$.

### Problem Classification

This optimization problem is **non-convex and combinatorially hard** in its global form.
Finding the globally optimal tree structure is NP-hard, as it would require evaluating all
possible recursive partitionings of the feature space simultaneously.

CART addresses this with a **greedy approximation**: rather than optimizing the full tree
jointly, it solves one local minimization problem at each node independently, choosing the
single best split at that level before recursing. This makes the algorithm:
- **Locally optimal** at each node, but not guaranteed globally optimal
- **Efficiently solvable**: for $p$ features and $n$ samples, evaluating all candidate
  thresholds requires $O(pn \log n)$ operations per node (sorting each feature once), making
  the full tree tractable in practice
- **A discrete optimization problem**: the search space for $\theta$ is finite — we only
  need to evaluate midpoints between consecutive unique values of each feature, as no other
  threshold would change the partition

### Connection to Information Theory

The Gini impurity minimization is closely related to minimizing **classification error**.
Minimizing weighted Gini at each split is equivalent to maximizing the **Gini gain**:


$$\Delta\text{Gini} = \text{Gini}(S_t) - \left(\frac{n_L}{n_t} \cdot \text{Gini}(S_L) + \frac{n_R}{n_t} \cdot \text{Gini}(S_R)\right)$$

An alternative criterion is **information gain** (entropy reduction), which uses
$H(S) = -\sum_k p_k \log p_k$ in place of Gini. Both criteria tend to produce
similar trees in practice; Gini is computationally preferred as it avoids the logarithm.


### Stopping Criteria as Constraints

The greedy minimization is subject to the following **constraints** that define when a node
becomes a leaf rather than splitting further:

| Constraint | Effect |
|---|---|
| $\text{depth}(t) \geq d_{\max}$ | Limits tree complexity; controls overfitting |
| $n_t < n_{\min\text{-split}}$ | Prevents splits on very small nodes |
| $n_L < n_{\min\text{-leaf}}$ or $n_R < n_{\min\text{-leaf}}$ | Ensures minimum child size |
| $\|C_t\| = 1$ (pure node) | Natural termination; no gain possible |
| $\Delta\text{Gini} = 0$ | No informative split exists |

In our implementation, $d_{\max} = 14$ was selected via validation set search (Section 7),
balancing the bias-variance trade-off observed in the tuning curve.

### Prediction as Tree Traversal

At inference time, each sample $x$ is routed from root to leaf by evaluating the learned
split rules sequentially:

$$\hat{y} = \arg\max_k \, \mathbb{1}[\text{sample reaches leaf } \ell] \cdot \text{count}(k, \ell)$$

Each leaf $\ell$ predicts the **majority class** among training samples that reached it —
a simple, interpretable, and closed-form prediction rule requiring no further optimization.

##4. DECISION TREE IMPLEMENTATION FROM SCRATCH

We implement a CART (Classification And Regression Trees) decision tree.

Core optimization problem:

At each internal node we solve:
argmin_{feature j, threshold t}  weighted_gini(left_split) + weighted_gini(right_split)

This is a greedy, locally optimal binary split search over all features and all candidate thresholds — the global problem is NP-hard, so CART uses this greedy recursive partitioning as an approximation.

Gini Impurity for a node with class proportions p_k:
        Gini(node) = 1 - Σ_k  p_k²

A pure node has Gini = 0; the worst case (uniform distribution) approaches 1.

In [ ]:
# ── 4.1  Node data structure ──────────────────────────────────────────────────
class Node:
    """
    Represents a single node in the decision tree.
    Internal nodes store the split rule (feature index + threshold).
    Leaf nodes store the predicted class label (majority vote).
    """
    def __init__(self):
        self.feature_index = None   # Index of the feature used to split
        self.threshold     = None   # Threshold value for the split
        self.left          = None   # Left child Node  (feature_value <= threshold)
        self.right         = None   # Right child Node (feature_value >  threshold)
        self.is_leaf       = False  # True when this node makes a prediction
        self.predicted_class = None # Majority class label (leaves only)
        self.gini          = None   # Gini impurity at this node (for reporting)
        self.n_samples     = None   # Number of training samples that reached here
        self.class_counts  = None   # Class distribution at this node

In [ ]:
# ── 4.2  Gini impurity helper ──────────────────────────────────────────────────
def gini_impurity(y):
    """
    Compute the Gini impurity of a label array.

    Gini(y) = 1 - Σ_k (count_k / n)²

    Parameters
    ----------
    y : 1-D numpy array of integer class labels

    Returns
    -------
    float in [0, 1]  — 0 means perfectly pure node
    """
    n = len(y)
    if n == 0:
        return 0.0
    # Count occurrences of each unique class
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / n
    return 1.0 - np.sum(probabilities ** 2)

In [ ]:
# ── 4.3  Best split finder (core optimization step) ───────────────────────────
def find_best_split(X, y, min_samples_split):
    """
    Search over all features and candidate thresholds for the split
    that minimises the weighted Gini impurity of the two child nodes.

    Objective:
        min_{j, t}  (n_L/n) * Gini(y_L) + (n_R/n) * Gini(y_R)

    where  y_L = y[X[:,j] <= t]  and  y_R = y[X[:,j] > t]

    Parameters
    ----------
    X               : 2-D array (n_samples, n_features)
    y               : 1-D array (n_samples,)
    min_samples_split : int — don't split if fewer samples than this

    Returns
    -------
    dict with keys 'feature_index', 'threshold', 'weighted_gini',
                   'left_mask', 'right_mask'
    OR None if no valid split exists.
    """
    n_samples, n_features = X.shape
    if n_samples < min_samples_split:
        return None

    best_gini    = np.inf   # Track lowest weighted Gini seen so far
    best_feature = None
    best_thresh  = None
    best_left    = None
    best_right   = None

    parent_gini = gini_impurity(y)   # Impurity before splitting

    for feature_idx in range(n_features):
        col = X[:, feature_idx]
        # Candidate thresholds: midpoints between consecutive sorted unique values.
        # Using unique values avoids evaluating duplicate thresholds, which
        # dramatically speeds up the search on binary (0/1) features.
        unique_vals = np.unique(col)
        if len(unique_vals) == 1:
            continue   # No split possible for a constant feature
        thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0

        for thresh in thresholds:
            left_mask  = col <= thresh
            right_mask = ~left_mask

            n_left  = left_mask.sum()
            n_right = right_mask.sum()

            # Both children must be non-empty
            if n_left == 0 or n_right == 0:
                continue

            # Weighted Gini of the proposed split
            w_gini = (n_left  / n_samples) * gini_impurity(y[left_mask]) + \
                     (n_right / n_samples) * gini_impurity(y[right_mask])

            if w_gini < best_gini:
                best_gini    = w_gini
                best_feature = feature_idx
                best_thresh  = thresh
                best_left    = left_mask
                best_right   = right_mask

    # If no improvement at all, do not split
    if best_feature is None:
        return None

    return {
        "feature_index" : best_feature,
        "threshold"     : best_thresh,
        "weighted_gini" : best_gini,
        "left_mask"     : best_left,
        "right_mask"    : best_right,
    }

In [ ]:
# ── 4.4  Recursive tree builder ────────────────────────────────────────────────
def build_tree(X, y, depth=0, max_depth=None,
               min_samples_split=2, min_samples_leaf=1):
    """
    Recursively grow a decision tree node using CART.

    Stopping criteria (any one triggers a leaf):
      1. Maximum depth reached.
      2. All labels in this node are identical (pure leaf).
      3. Fewer than min_samples_split samples remain.
      4. No valid split was found.
      5. A proposed child would have fewer than min_samples_leaf samples.

    Parameters
    ----------
    X, y              : arrays at this node
    depth             : current tree depth (root = 0)
    max_depth         : int or None — cap on tree depth
    min_samples_split : int — minimum samples required to attempt a split
    min_samples_leaf  : int — minimum samples required in each child

    Returns
    -------
    Node object (the root of the (sub)tree)
    """
    node = Node()
    node.n_samples    = len(y)
    node.gini         = gini_impurity(y)
    classes, counts   = np.unique(y, return_counts=True)
    node.class_counts = dict(zip(classes, counts))
    # Majority vote: the most frequent class is the prediction
    node.predicted_class = classes[np.argmax(counts)]

    # ── Stopping condition checks ──────────────────────────────────────────────
    # (1) Max depth reached
    if max_depth is not None and depth >= max_depth:
        node.is_leaf = True
        return node
    # (2) Pure node — all samples belong to one class
    if len(classes) == 1:
        node.is_leaf = True
        return node
    # (3) Too few samples to consider splitting
    if len(y) < min_samples_split:
        node.is_leaf = True
        return node

    # ── Find the best split ────────────────────────────────────────────────────
    split = find_best_split(X, y, min_samples_split)

    # (4) No valid split found (e.g., all features constant)
    if split is None:
        node.is_leaf = True
        return node

    # (5) Enforce min_samples_leaf — both children must be large enough
    n_left  = split["left_mask"].sum()
    n_right = split["right_mask"].sum()
    if n_left < min_samples_leaf or n_right < min_samples_leaf:
        node.is_leaf = True
        return node

    # ── Record the split and recurse ───────────────────────────────────────────
    node.feature_index = split["feature_index"]
    node.threshold     = split["threshold"]

    node.left  = build_tree(X[split["left_mask"]],  y[split["left_mask"]],
                            depth + 1, max_depth, min_samples_split, min_samples_leaf)
    node.right = build_tree(X[split["right_mask"]], y[split["right_mask"]],
                            depth + 1, max_depth, min_samples_split, min_samples_leaf)
    return node

In [ ]:
# ── 4.5  Prediction helpers ────────────────────────────────────────────────────
def predict_single(node, x):
    """
    Traverse the tree for a single sample x and return the predicted class.

    Traversal rule:
      x[feature_index] <= threshold  →  go left
      x[feature_index] >  threshold  →  go right
    """
    if node.is_leaf:
        return node.predicted_class
    if x[node.feature_index] <= node.threshold:
        return predict_single(node.left, x)
    else:
        return predict_single(node.right, x)

def predict(node, X):
    """
    Predict class labels for all rows of X by calling predict_single row-by-row.

    Parameters
    ----------
    node : root Node of a trained tree
    X    : 2-D numpy array (n_samples, n_features)

    Returns
    -------
    1-D numpy array of predicted integer class labels
    """
    return np.array([predict_single(node, x) for x in X])

In [ ]:
# ── 4.6  Tree introspection utilities ─────────────────────────────────────────
def tree_depth(node):
    """Return the maximum depth of the tree (root = depth 0)."""
    if node is None or node.is_leaf:
        return 0
    return 1 + max(tree_depth(node.left), tree_depth(node.right))

def count_leaves(node):
    """Count the total number of leaf nodes in the tree."""
    if node is None:
        return 0
    if node.is_leaf:
        return 1
    return count_leaves(node.left) + count_leaves(node.right)

def count_nodes(node):
    """Count ALL nodes (internal + leaves) in the tree."""
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

def feature_importances(node, n_features, X, y):
    """
    Compute feature importance scores based on total Gini reduction.

    For each split node, the importance contribution is:
        importance_j += (n_node/n_total) * [Gini(node) - (n_L/n_node)*Gini(L) - (n_R/n_node)*Gini(R)]

    This mirrors sklearn's 'gini importance' / 'mean decrease impurity'.
    importances are then normalised to sum to 1.
    """
    importances = np.zeros(n_features)
    n_total = len(y)

    def traverse(node):
        if node is None or node.is_leaf:
            return
        j   = node.feature_index
        n   = node.n_samples
        g   = node.gini
        n_L = node.left.n_samples
        n_R = node.right.n_samples
        g_L = node.left.gini
        g_R = node.right.gini

        # Weighted Gini decrease caused by this split
        gain = (n / n_total) * (g - (n_L / n) * g_L - (n_R / n) * g_R)
        importances[j] += gain

        traverse(node.left)
        traverse(node.right)

    traverse(node)
    total = importances.sum()
    if total > 0:
        importances /= total    # Normalise so they sum to 1
    return importances

##5. TRAIN THE CUSTOM DECISION TREE

We train with max_depth=10.  Shallow trees underfit; very deep trees overfit.

Depth 10 is a reasonable starting point; we tune it in Section 7.

In [ ]:
import time

MAX_DEPTH = 10   # Hyperparameter — tuned via validation set in Section 7

print(f"Training custom Decision Tree (max_depth={MAX_DEPTH}) ...")

start = time.time()
custom_tree = build_tree(
    X_train, y_train,
    max_depth=MAX_DEPTH,
    min_samples_split=2,
    min_samples_leaf=1,
)
elapsed = time.time() - start

print(f"Training complete in {elapsed:.1f}s")
print(f"  Tree depth   : {tree_depth(custom_tree)}")
print(f"  Total nodes  : {count_nodes(custom_tree)}")
print(f"  Leaf nodes   : {count_leaves(custom_tree)}")

## Custom Tree Training — Initial Results (max_depth = 10)

The from-scratch CART implementation successfully trained in **46.4 seconds** — considerably
slower than sklearn's C-optimized implementation (0.13s), but expected for a pure Python
implementation. The tree reached its full allowed depth of 10, producing **821 nodes** with
**411 leaf nodes**, meaning the tree learned 411 distinct decision regions in feature space.

The training accuracy of **0.8392** vs. validation accuracy of **0.7623** reveals a moderate
degree of overfitting — the tree memorizes some training-set patterns that do not generalize.
This gap motivates the hyperparameter tuning in Section 7 to find the optimal depth.

In [ ]:
# ── 5.1  Train & Validation accuracy for the custom tree ──────────────────────
y_train_pred_custom = predict(custom_tree, X_train)
y_val_pred_custom   = predict(custom_tree, X_val)

train_acc_custom = accuracy_score(y_train, y_train_pred_custom)
val_acc_custom   = accuracy_score(y_val,   y_val_pred_custom)

print(f"Custom Tree — Training Accuracy  : {train_acc_custom:.4f}")
print(f"Custom Tree — Validation Accuracy: {val_acc_custom:.4f}")

##6. SKLEARN BLACK-BOX DECISION TREE (BENCHMARK)

sklearn's DecisionTreeClassifier also uses CART with Gini impurity, but its C-optimised implementation runs orders of magnitude faster.

Using identical hyperparameters lets us verify correctness of our scratch code.


In [ ]:
sklearn_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=MAX_DEPTH,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=RANDOM_SEED,
)

start = time.time()
sklearn_tree.fit(X_train, y_train)
elapsed_sk = time.time() - start

y_train_pred_sk = sklearn_tree.predict(X_train)
y_val_pred_sk   = sklearn_tree.predict(X_val)

train_acc_sk = accuracy_score(y_train, y_train_pred_sk)
val_acc_sk   = accuracy_score(y_val,   y_val_pred_sk)

print(f"sklearn Tree — Training Accuracy  : {train_acc_sk:.4f}  (fit time: {elapsed_sk:.2f}s)")
print(f"sklearn Tree — Validation Accuracy: {val_acc_sk:.4f}")

print(f"\n--- Accuracy Delta (Custom vs sklearn) ---")
print(f"  Train : {abs(train_acc_custom - train_acc_sk):.5f}")
print(f"  Val   : {abs(val_acc_custom   - val_acc_sk):.5f}")
print("\nA near-zero delta confirms our scratch implementation is correct.")

## Custom vs. sklearn Benchmark Comparison

The accuracy delta between our scratch implementation and sklearn is essentially zero on the
training set (**0.00000**) and negligible on the validation set (**0.00088** — less than 0.1%).
This near-perfect agreement confirms that our CART implementation is **mathematically correct**:
both implementations use identical Gini-based greedy split selection, the same stopping
criteria, and majority-vote leaf prediction.

The key practical difference is **speed**: sklearn fits in 0.13 seconds versus 46.4 seconds
for our implementation. This gap arises because sklearn's backend is written in Cython/C with
pre-sorted feature arrays and optimized memory access patterns, while our implementation
iterates over thresholds in pure Python. For the purposes of this project, correctness is
verified and the speed trade-off is acceptable.

##7. HYPERPARAMETER TUNING — MAX DEPTH

We sweep max_depth from 1 to 20 using the validation set. The validation set acts as a proxy for unseen data during model selection.

Final test set is held out until Section 8.


In [ ]:
depths         = list(range(1, 21))
val_accs_sk    = []    # sklearn validation accuracies per depth
train_accs_sk  = []    # sklearn training accuracies per depth

print("Sweeping max_depth using sklearn (fast surrogate for tuning) ...")

for d in depths:
    clf = DecisionTreeClassifier(
        criterion="gini", max_depth=d, random_state=RANDOM_SEED
    )
    clf.fit(X_train, y_train)
    train_accs_sk.append(accuracy_score(y_train, clf.predict(X_train)))
    val_accs_sk.append(accuracy_score(y_val, clf.predict(X_val)))

best_depth = depths[np.argmax(val_accs_sk)]
print(f"\nBest max_depth = {best_depth}  (val accuracy = {max(val_accs_sk):.4f})")

In [ ]:
# ── 7.1  Plot depth vs accuracy ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_accs_sk, "o-",  color="steelblue", label="Train Accuracy")
ax.plot(depths, val_accs_sk,   "s--", color="tomato",    label="Validation Accuracy")
ax.axvline(best_depth, color="green", linestyle=":", linewidth=2,
           label=f"Best depth = {best_depth}")
ax.set_xlabel("Max Depth", fontsize=12)
ax.set_ylabel("Accuracy",  fontsize=12)
ax.set_title("Decision Tree: Training vs Validation Accuracy by Max Depth",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.set_xticks(depths)
plt.tight_layout()
plt.savefig("depth_tuning_curve.png", dpi=150)
plt.show()

## Hyperparameter Tuning — Max Depth Analysis

The learning curve reveals a classic **bias-variance trade-off**:
- At shallow depths (1–4), both training and validation accuracy are low — the model
  **underfits**, as simple splits cannot capture the complexity of 7-class forest cover
  classification.
- From depth 5 onward, training accuracy climbs steadily toward 1.0 as the tree memorizes
  more training patterns.
- Validation accuracy peaks at **depth 14 (0.7888)** and then plateaus or marginally declines,
  indicating the onset of **overfitting** — additional splits capture noise rather than signal.
- The green dashed line at depth 14 marks our selected hyperparameter, balancing
  expressiveness and generalization.

In [ ]:
# ── 7.2  Re-train custom tree with best depth ──────────────────────────────────
print(f"\nRe-training custom tree with best max_depth = {best_depth} ...")
start = time.time()
best_custom_tree = build_tree(
    X_train, y_train,
    max_depth=best_depth,
    min_samples_split=2,
    min_samples_leaf=1,
)
elapsed_retrain = time.time() - start
print(f"Done in {elapsed_retrain:.1f}s")
print(f"  Depth : {tree_depth(best_custom_tree)}")
print(f"  Nodes : {count_nodes(best_custom_tree)}")
print(f"  Leaves: {count_leaves(best_custom_tree)}")

## Re-trained Custom Tree at Optimal Depth

With max_depth = 14, the tree grew to **1,959 nodes and 980 leaf nodes** — more than doubling
the complexity of the initial depth-10 tree. Each leaf node represents a unique decision region
carved out by up to 14 sequential binary splits. The training time of **61.8 seconds** reflects
the increased tree complexity, as deeper nodes operate on smaller and more refined subsets of
the data where threshold search is more exhaustive.

## 8. FINAL EVALUATION ON HELD-OUT TEST SET

We now evaluate BOTH implementations on the test set — the first time the test set is touched.  

This gives an unbiased estimate of generalisation.


In [ ]:
# ── Custom tree ───────────────────────────────────────────────────────────────
y_test_pred_custom = predict(best_custom_tree, X_test)
test_acc_custom    = accuracy_score(y_test, y_test_pred_custom)

# ── sklearn tree (same best depth) ────────────────────────────────────────────
best_sklearn_tree = DecisionTreeClassifier(
    criterion="gini", max_depth=best_depth, random_state=RANDOM_SEED
)
best_sklearn_tree.fit(X_train, y_train)
y_test_pred_sk = best_sklearn_tree.predict(X_test)
test_acc_sk    = accuracy_score(y_test, y_test_pred_sk)

print("=" * 55)
print(f"  TEST SET RESULTS  (max_depth = {best_depth})")
print("=" * 55)
print(f"  Custom tree test accuracy : {test_acc_custom:.4f}")
print(f"  sklearn tree test accuracy: {test_acc_sk:.4f}")
print(f"  Accuracy delta            : {abs(test_acc_custom - test_acc_sk):.5f}")

## Final Test Set Evaluation

Both implementations achieve approximately **79.5–79.6% test accuracy**, confirming that the
optimal depth identified on the validation set generalizes well to truly unseen data.

**Per-class performance highlights from the classification reports:**
- **Cottonwood/Willow (Type 4)** achieves the highest F1-score (~0.92), likely because its
  distinctly low elevation signature (observed in EDA) makes it easily separable.
- **Krummholz (Type 7)** also performs strongly (F1 ≈ 0.91), consistent with its exclusive
  occupancy of the highest elevations.
- **Lodgepole Pine (Type 2)** is the hardest class to classify (recall ≈ 0.55, F1 ≈ 0.61),
  likely because it occupies a broad mid-elevation range that overlaps heavily with Spruce/Fir
  (Type 1) and Ponderosa Pine (Type 3).
- The **macro-averaged F1 of 0.79** across both implementations is consistent, confirming
  that neither implementation is biased toward any subset of classes.

The custom and sklearn trees agree on predictions for the vast majority of test samples,
with a delta of only **0.00088** — one or two samples classified differently out of 2,268.

In [ ]:
# ── 8.1  Classification Report — Custom Tree ───────────────────────────────────
print("\n=== Custom Tree — Full Classification Report ===")
print(classification_report(y_test, y_test_pred_custom,
                             target_names=class_names))

In [ ]:
# ── 8.2  Classification Report — sklearn Tree ─────────────────────────────────
print("\n=== sklearn Tree — Full Classification Report ===")
print(classification_report(y_test, y_test_pred_sk,
                             target_names=class_names))

In [ ]:
# ── 8.3  Confusion Matrices side-by-side ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, preds, title in zip(
    axes,
    [y_test_pred_custom, y_test_pred_sk],
    ["Custom Decision Tree", "sklearn Decision Tree"]
):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues", xticks_rotation=45)
    ax.set_title(title, fontsize=13, fontweight="bold")

plt.suptitle("Confusion Matrices — Test Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()

## Confusion Matrix Analysis

The confusion matrices for both implementations are nearly identical, reinforcing the
correctness of our scratch implementation. Key patterns:
- The **diagonal dominates** in all classes, indicating generally reliable predictions.
- The most frequent off-diagonal errors occur between **Spruce/Fir (1) and Lodgepole Pine (2)**,
  which share similar mid-to-high elevation ranges and terrain characteristics.
- **Ponderosa Pine (3) and Douglas-fir (6)** also show mutual confusion (~44–53 samples
  misclassified in each direction), consistent with their overlapping elevation and soil profiles.
- **Cottonwood/Willow (4)** and **Krummholz (7)** have the cleanest rows and columns,
  reflecting their ecologically distinct niches that the tree captures well with just a few
  high-level splits.

In [ ]:
# ── 8.4  Per-class accuracy bar chart ─────────────────────────────────────────
cm_custom = confusion_matrix(y_test, y_test_pred_custom)
cm_sk     = confusion_matrix(y_test, y_test_pred_sk)

per_class_acc_custom = cm_custom.diagonal() / cm_custom.sum(axis=1)
per_class_acc_sk     = cm_sk.diagonal()     / cm_sk.sum(axis=1)

x = np.arange(len(class_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, per_class_acc_custom, width, label="Custom Tree", color="steelblue", edgecolor="black")
ax.bar(x + width/2, per_class_acc_sk,     width, label="sklearn Tree", color="tomato",    edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=30, ha="right")
ax.set_ylabel("Per-class Accuracy")
ax.set_title("Per-Class Accuracy: Custom vs sklearn Decision Tree",
             fontsize=13, fontweight="bold")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig("per_class_accuracy.png", dpi=150)
plt.show()

##9. FEATURE IMPORTANCE ANALYSIS


Feature importances = total normalised Gini reduction attributed to each feature.

High importance → the feature was used for many high-gain splits.


In [ ]:
# ── Custom tree importances ────────────────────────────────────────────────────
custom_importances = feature_importances(best_custom_tree, X_train.shape[1],
                                         X_train, y_train)

# ── sklearn tree importances ───────────────────────────────────────────────────
sklearn_importances = best_sklearn_tree.feature_importances_

# ── Show top-15 features from custom tree ─────────────────────────────────────
top_n = 15
top_idx_custom = np.argsort(custom_importances)[::-1][:top_n]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, importances, title, color in zip(
    axes,
    [custom_importances, sklearn_importances],
    ["Custom Tree — Top 15 Features", "sklearn Tree — Top 15 Features"],
    ["steelblue", "tomato"]
):
    top_idx = np.argsort(importances)[::-1][:top_n]
    ax.barh(
        [feature_names[i] for i in top_idx][::-1],
        importances[top_idx][::-1],
        color=color, edgecolor="black"
    )
    ax.set_xlabel("Normalised Importance (Gini Reduction)")
    ax.set_title(title, fontsize=12, fontweight="bold")

plt.suptitle("Feature Importances: Custom vs sklearn Decision Tree",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("feature_importances.png", dpi=150)
plt.show()

## Feature Importance Insights

Both the custom and sklearn trees rank features nearly identically, further validating our
implementation:
- **Elevation** is overwhelmingly the most important feature, responsible for the largest
  single share of total Gini reduction. This aligns with the EDA observation that elevation
  separates classes most cleanly (e.g., Krummholz at high altitude, Cottonwood/Willow at low).
- **Horizontal_Distance_To_Roadways** and **Hillshade_9am** rank highly, suggesting that
  proximity to infrastructure and morning solar exposure are strong secondary discriminators.
- **Horizontal_Distance_To_Fire_Points** and **Horizontal_Distance_To_Hydrology** contribute
  meaningfully, capturing ecological dependencies on water and fire history.
- The **40 Soil Type binary features** contribute comparatively little individually, though
  they collectively provide refinement at deeper tree levels for harder-to-separate classes.
- The `Id` column appearing in the custom tree's top features is a data artifact — `Id` was
  dropped at loading time, so if it appears, verify the `df.drop` step executed correctly.
  The sklearn importances (which correctly exclude it) should be treated as the reference.

##10. TREE STRUCTURE VISUALISATION

We visualise the sklearn tree (same logic as ours) at a shallow depth for interpretability.  Deep trees are unreadable, so we show max_depth=4.



In [ ]:
shallow_tree = DecisionTreeClassifier(
    criterion="gini", max_depth=4, random_state=RANDOM_SEED
)
shallow_tree.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(28, 10))
plot_tree(
    shallow_tree,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,           # Colour nodes by majority class
    impurity=True,         # Show Gini at each node
    rounded=True,
    fontsize=8,
    ax=ax,
)
ax.set_title("Decision Tree Structure (max_depth=4 for readability)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("tree_structure.png", dpi=150)
plt.show()

## Tree Structure Interpretation

The shallow tree (max_depth = 4) reveals the top-level decision logic learned by CART:
- The **root node splits on Elevation**, confirming it as the single most discriminative
  feature. Samples above the threshold tend toward high-altitude types (Krummholz), while
  those below branch further.
- **Second-level splits** use Elevation again and introduce Hillshade and Distance features,
  refining within the elevation bands established at the root.
- By depth 4, the tree has already created reasonably pure regions for the ecologically
  extreme classes (Krummholz, Cottonwood/Willow), while mid-elevation classes (Lodgepole
  Pine, Spruce/Fir) remain mixed and require the additional 10 levels of the full model.
- The color coding shows that **blue-toned nodes** (majority Spruce/Fir or Lodgepole Pine)
  dominate the left subtree (lower elevation), while **orange/green nodes** (Krummholz,
  Douglas-fir) appear in the right high-elevation subtree.

##11. SUMMARY TABLE — CUSTOM vs SKLEARN

Collect all key metrics in one place for easy reporting.

In [ ]:
summary = {
    "Metric"                 : ["Train Accuracy", "Validation Accuracy",
                                "Test Accuracy", "Max Depth",
                                "Total Nodes", "Leaf Nodes"],
    "Custom Tree"            : [
        f"{accuracy_score(y_train, predict(best_custom_tree, X_train)):.4f}",
        f"{accuracy_score(y_val,   predict(best_custom_tree, X_val)):.4f}",
        f"{test_acc_custom:.4f}",
        str(tree_depth(best_custom_tree)),
        str(count_nodes(best_custom_tree)),
        str(count_leaves(best_custom_tree)),
    ],
    "sklearn Tree"           : [
        f"{accuracy_score(y_train, best_sklearn_tree.predict(X_train)):.4f}",
        f"{accuracy_score(y_val,   best_sklearn_tree.predict(X_val)):.4f}",
        f"{test_acc_sk:.4f}",
        str(best_sklearn_tree.get_depth()),
        str(best_sklearn_tree.tree_.node_count),
        str(best_sklearn_tree.get_n_leaves()),
    ],
}

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df

## Summary and Conclusions

**Key takeaways:**
1. Our from-scratch CART implementation **correctly replicates sklearn's results**, with a test
   accuracy delta of only 0.0009 (less than 0.1%), validating the correctness of our Gini-based
   greedy split optimization.
2. Both trees show a notable **train-to-test accuracy gap** (~0.94 train vs. ~0.80 test),
   indicating moderate overfitting even at the tuned depth of 14. The tree memorizes fine-grained
   training patterns that do not fully generalize — an inherent limitation of a single unpruned
   decision tree.
3. The final test accuracy of ~**79.5–79.6%** on a 7-class balanced problem is a solid result
   for a single decision tree, with Cottonwood/Willow and Krummholz classified near-perfectly
   due to their distinct elevation signatures identified in the EDA.
4. The two implementations produce **slightly different tree structures** (1,959 vs. 1,951 nodes;
   980 vs. 976 leaves) despite identical hyperparameters and nearly identical accuracy. This is
   expected: when two candidate splits produce equal or near-equal weighted Gini values, tie-breaking
   behavior differs between our Python implementation and sklearn's Cython backend, leading to
   slightly divergent branching at deeper levels — without meaningfully affecting predictive performance.
5. The primary limitation remains **Lodgepole Pine classification (recall ≈ 55%)**, which overlaps
   heavily with neighboring species in feature space. This is an inherent challenge for a single
   decision tree that ensemble methods such as Random Forests would address by averaging across
   many decorrelated trees to reduce variance.
6. The main implementation challenge was **computational efficiency**: our pure Python threshold
   search trained in ~62 seconds versus sklearn's ~0.13 seconds — roughly a 475× slowdown —
   highlighting the engineering value of compiled, optimized implementations for practical use.


---

---------------
:)